# Challenge 3: Quantum Spy Network ★★☆

**The mission:** Alice wants to send Bob a secret message. First they need a shared secret **key**, but Eve is listening on the line. You'll build **BB84**, the first quantum key distribution protocol (invented by Bennett and Brassard in 1984), and watch quantum physics catch Eve red-handed.

**Quantum ideas:** measurement disturbs a qubit; measuring in different "bases."
**Best for:** cybersecurity and CS folks, and anyone who likes spy stories.
**Time plan:** Missions 1 to 5 take about 90 minutes. Spend the rest on "Make it yours."

## The big idea

Alice can write a bit onto a qubit in one of two ways, called **bases**:

| Basis | To write bit 0 | To write bit 1 |
| --- | --- | --- |
| `Z` ("straight") | do nothing | X |
| `X` ("diagonal") | H | X, then H |

To read a qubit, Bob also picks a basis. For `Z`, he just measures. For `X`, he applies H and then measures.

- **Same basis as Alice:** Bob reads her bit exactly.
- **Different basis:** Bob gets a random coin flip.

Eve has the same problem. She doesn't know which basis Alice used, so when she peeks, she often guesses wrong and scrambles the qubit. That's how Alice and Bob catch her.

The setup cell gives you some helpers: `quantum_random_bits(n)`, `random_bases(n)`, `text_to_bits(text)`, and `bits_to_text(bits)`.

In [ ]:
# Using Google Colab? Delete the "#" at the start of the next line, then run this cell once.
# !pip install -q "qiskit[visualization]"

In [ ]:
# SETUP: run this cell first (click it, then press Shift+Enter).
# You don't need to read or change anything in this cell.

from qiskit import QuantumCircuit
from qiskit.primitives import StatevectorSampler
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_histogram, plot_bloch_multivector
from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import math, random, qiskit

_sampler = StatevectorSampler()

def _needs_measure(qc):
    if qc.num_clbits == 0:
        raise ValueError("Your circuit has no measurements yet. Add qc.measure_all() as the last step of your circuit.")

def run(qc, shots=1000):
    """Run a circuit many times. Returns counts, e.g. {'0': 507, '1': 493}."""
    _needs_measure(qc)
    return _sampler.run([qc], shots=shots).result()[0].join_data().get_counts()

def run_once(qc):
    """Run a circuit one time. Returns a single result as a string, e.g. '101'."""
    _needs_measure(qc)
    return _sampler.run([qc], shots=1).result()[0].join_data().get_bitstrings()[0]

def run_many(qc, shots):
    """Run a circuit `shots` times. Returns every result in order, e.g. ['0', '1', '1', ...]."""
    _needs_measure(qc)
    return _sampler.run([qc], shots=shots).result()[0].join_data().get_bitstrings()

def probabilities(qc):
    """Exact chance of each result, with no randomness. Measurements at the end are ignored."""
    sv = Statevector(qc.remove_final_measurements(inplace=False))
    return {str(k): round(float(v), 4) for k, v in sv.probabilities_dict().items() if v > 1e-9}

def _close(got, want, tol=0.02):
    return all(abs(got.get(k, 0) - want.get(k, 0)) <= tol for k in set(got) | set(want))

def _report(ok, yes, hint):
    if ok:
        print("\u2705 " + yes)
    else:
        print("\u274c Not yet. Hint: " + hint)

def _check_probs(qc, want, yes, hint):
    try:
        ok = _close(probabilities(qc), want)
    except Exception:
        ok = False
    _report(ok, yes, hint)

# ---- checks for this notebook ----
def quantum_random_bits(n):
    """n truly random bits (0 or 1) from a quantum coin."""
    coin = QuantumCircuit(1); coin.h(0); coin.measure_all()
    return [int(b) for b in run_many(coin, n)]

def random_bases(n):
    """n random bases, each 'Z' or 'X', chosen by a quantum coin."""
    return ["Z" if b == 0 else "X" for b in quantum_random_bits(n)]

def text_to_bits(text):
    """'Hi' -> [0,1,0,0,1,0,0,0, 0,1,1,0,1,0,0,1] (8 bits per character)."""
    return [int(b) for ch in text for b in format(ord(ch), "08b")]

def bits_to_text(bits):
    """The reverse of text_to_bits."""
    chars = []
    for i in range(0, len(bits) - 7, 8):
        chars.append(chr(int("".join(str(b) for b in bits[i:i + 8]), 2)))
    return "".join(chars)

def check_mission_1(send_qubit):
    try:
        ok = True
        for bit in [0, 1]:
            for a in "ZX":
                for b in "ZX":
                    got = {send_qubit(bit, a, b) for _ in range(30)}
                    ok = ok and (got == {bit} if a == b else got == {0, 1})
    except Exception:
        ok = False
    _report(ok, "Mission 1 complete: same basis = same bit, different basis = coin flip.",
            "Alice: X if the bit is 1, then H if her basis is 'X'. Bob: H if his basis is 'X', then measure.")

def check_mission_2(alice_key, bob_key, N):
    ok = len(alice_key) > N // 4 and alice_key == bob_key
    _report(ok, f"Mission 2 complete: Alice and Bob share a {len(alice_key)}-bit secret key.",
            "keep position i only when alice_bases[i] == bob_bases[i]; use the same rule for both keys.")

def check_mission_3(send_with_eve):
    try:
        errors = kept = 0
        for _ in range(400):
            bit, a, e, b = random.randint(0, 1), random.choice("ZX"), random.choice("ZX"), random.choice("ZX")
            if a == b:
                kept += 1
                errors += send_with_eve(bit, a, e, b) != bit
        ok = 0.12 < errors / kept < 0.38
    except Exception:
        ok = False
    _report(ok, "Mission 3 complete: Eve's spying causes about 25% errors.",
            "Eve doesn't know Alice's bit or basis. She must READ the qubit with send_qubit, then re-send in her own basis.")

def check_mission_4(eve_detected):
    try:
        k = [0, 1] * 10
        bad = [1 - b if i < 5 else b for i, b in enumerate(k)]
        ok = eve_detected(k, k) == (False, 0.0) and eve_detected(k, bad) == (True, 0.25)
    except Exception:
        ok = False
    _report(ok, "Mission 4 complete: your Eve detector works.",
            "count positions where alice_sample[i] != bob_sample[i], then divide by sample_size.")

def check_mission_5(message, encrypted, decrypted):
    ok = encrypted != text_to_bits(message) and bits_to_text(decrypted) == message
    _report(ok, "Mission 5 complete: secret message delivered!",
            "encrypt with m ^ k for each message bit m and key bit k. Decrypt the same way with Bob's key.")

print(f"\u2705 Setup complete (Qiskit {qiskit.__version__}). You're ready to go!")

## Mission 1: Send one qubit

**TODO:** Finish `send_qubit` using the table above. There are two TODOs: one for Alice, one for Bob.

In [ ]:
def send_qubit(bit, alice_basis, bob_basis):
    """Alice writes `bit` in alice_basis, Bob reads it in bob_basis. Returns Bob's bit (0 or 1)."""
    qc = QuantumCircuit(1)

    # --- Alice writes the bit ---
    # TODO: if bit is 1, apply X. Then, if alice_basis is "X", apply H.

    # --- Bob reads the bit ---
    # TODO: if bob_basis is "X", apply H before measuring.

    qc.measure_all()
    return int(run_once(qc))

# Try every combination 10 times
for bit in [0, 1]:
    for a in ["Z", "X"]:
        for b in ["Z", "X"]:
            results = [send_qubit(bit, a, b) for i in range(10)]
            print(f"Alice sends {bit} in {a}, Bob reads in {b}: {results}")

In [ ]:
check_mission_1(send_qubit)

## Mission 2: Build a shared key (no Eve yet)

Here is the protocol:

1. Alice picks random bits and random bases, and sends one qubit per bit.
2. Bob reads each qubit in a random basis.
3. They announce their **bases** publicly, but never their bits.
4. They keep only the positions where their bases matched. Those bits are the secret key.

**TODO:** Build `alice_key` and `bob_key` from the positions where the bases match.

In [ ]:
N = 64
alice_bits = quantum_random_bits(N)
alice_bases = random_bases(N)
bob_bases = random_bases(N)

bob_bits = [send_qubit(alice_bits[i], alice_bases[i], bob_bases[i]) for i in range(N)]

# TODO: keep only positions where alice_bases[i] == bob_bases[i]
alice_key = []
bob_key = []

print("Alice's key:", alice_key)
print("Bob's key:  ", bob_key)
print(f"Kept {len(alice_key)} of {N} bits. Keys match: {alice_key == bob_key}")

In [ ]:
check_mission_2(alice_key, bob_key, N)

About half the bits survive, because the bases match about half the time.

## Mission 3: Enter Eve

Eve cuts into the line. For each qubit she:

1. Reads Alice's qubit in a random basis of her own.
2. Sends Bob a fresh qubit with what she read, written in **her** basis.

Notice that Eve reading a qubit is exactly what `send_qubit` already does, with Eve as the receiver.

**TODO:** Finish `send_with_eve`. The placeholder lines let Eve cheat by knowing things she can't know; replace them.

In [ ]:
def send_with_eve(bit, alice_basis, eve_basis, bob_basis):
    # Step 1: Eve reads Alice's qubit in her own basis.
    # TODO: get eve_bit with send_qubit
    eve_bit = bit   # <- cheating: Eve can't know Alice's bit for free!

    # Step 2: Eve sends what she read to Bob, written in HER basis.
    # TODO: return Bob's bit with send_qubit
    return send_qubit(eve_bit, alice_basis, bob_basis)   # <- cheating: Eve doesn't know Alice's basis!

N = 200
alice_bits = quantum_random_bits(N)
alice_bases, eve_bases, bob_bases = random_bases(N), random_bases(N), random_bases(N)
bob_bits = [send_with_eve(alice_bits[i], alice_bases[i], eve_bases[i], bob_bases[i]) for i in range(N)]

keep = [i for i in range(N) if alice_bases[i] == bob_bases[i]]
errors = sum(1 for i in keep if alice_bits[i] != bob_bits[i])
print(f"Kept {len(keep)} bits. Errors: {errors} ({errors / len(keep):.0%})")

In [ ]:
check_mission_3(send_with_eve)

You should see roughly **25% errors**, even though Alice and Bob used the same basis. Why 25%? Half the time Eve guesses the wrong basis. When she does, Bob's result is a coin flip, so it's wrong half of those times. ½ × ½ = ¼.

Eve can't avoid this by copying the qubit and measuring the copy, because a rule of quantum physics (the **no-cloning theorem**) says unknown qubits can't be copied.

## Mission 4: Catch Eve

Alice and Bob sacrifice a few key bits: they compare them in public. Without Eve, these should match perfectly. Lots of mismatches means someone is listening.

**TODO:** Count the mismatches in the sample and compute `error_rate`.

In [ ]:
def eve_detected(alice_key, bob_key, sample_size=20, threshold=0.10):
    """Compare the first sample_size bits in public. Returns (detected, error_rate)."""
    alice_sample = alice_key[:sample_size]
    bob_sample = bob_key[:sample_size]
    # TODO: count positions where the samples differ, then divide by sample_size
    error_rate = 0.0
    return error_rate > threshold, error_rate

def run_bb84(n, eve_present):
    """Run the whole protocol with n qubits. Returns (alice_key, bob_key)."""
    alice_bits = quantum_random_bits(n)
    alice_bases, eve_bases, bob_bases = random_bases(n), random_bases(n), random_bases(n)
    if eve_present:
        bob_bits = [send_with_eve(alice_bits[i], alice_bases[i], eve_bases[i], bob_bases[i]) for i in range(n)]
    else:
        bob_bits = [send_qubit(alice_bits[i], alice_bases[i], bob_bases[i]) for i in range(n)]
    keep = [i for i in range(n) if alice_bases[i] == bob_bases[i]]
    return [alice_bits[i] for i in keep], [bob_bits[i] for i in keep]

for eve_present in [False, True]:
    alice_key, bob_key = run_bb84(200, eve_present)
    detected, rate = eve_detected(alice_key, bob_key)
    print(f"Eve on the line: {eve_present}  ->  error rate {rate:.0%}, Eve detected: {detected}")

In [ ]:
check_mission_4(eve_detected)

## Mission 5: Send the secret message

With a safe key, Alice can use a **one-time pad**: combine each message bit with a key bit using XOR (`^` in Python). XOR gives 1 when two bits differ and 0 when they match. Doing it twice undoes it: `(m ^ k) ^ k == m`. So encrypting and decrypting are the same step.

The key must be at least as long as the message (8 bits per character). Use plain letters, numbers, and punctuation.

**TODO:** Encrypt with Alice's key, then decrypt with Bob's key.

In [ ]:
message = "MEET AT THE LIBRARY"
message_bits = text_to_bits(message)

alice_key, bob_key = run_bb84(500, eve_present=False)
detected, rate = eve_detected(alice_key, bob_key)
alice_key, bob_key = alice_key[20:], bob_key[20:]    # throw away the 20 bits they compared in public
print(f"Eve detected: {detected} | key bits available: {len(alice_key)} | needed: {len(message_bits)}")

# TODO: Alice encrypts: XOR each message bit with her key bit
encrypted = message_bits   # <- not encrypted yet!
print("What Eve sees on the line:", "".join(str(b) for b in encrypted[:48]), "...")

# TODO: Bob decrypts: XOR each encrypted bit with his key bit
decrypted = encrypted
print("Bob reads:", bits_to_text(decrypted))

In [ ]:
check_mission_5(message, encrypted, decrypted)

## Make it yours

Pick an idea below or invent your own:

- **Sneaky Eve.** Eve only spies on some qubits, say 1 in 4. How does the error rate change? Graph error rate against how often Eve listens. How sneaky can she be and still get caught by a 20-bit check?
- **Noisy line vs. Eve.** Real fiber-optic lines flip some bits by accident. Add random noise, then pick a threshold that tells noise apart from an eavesdropper.
- **Spy case file.** Make a colorful table or chart showing each qubit's bit, bases, whether it was kept, and whether it had an error.
- **Smarter Eve.** Is Eve better off always using one basis? Test it and explain why.
- **Encrypted chat.** Wrap everything into a function that sends any message and reports whether the line was safe.

## Explain it

Answer in your own words and be ready to say them out loud.

1. Why do Alice and Bob throw away about half their bits?
2. Why does Eve cause about 25% errors?
3. Why can't Eve just copy each qubit and send the original along untouched?

Our answers: